# Step 2: Household environmental impact allocation based on socio-economic and demographic factors.

Before running this notebook, please refer to the [Step 2 README](../Step_2/README.md) for instructions on the required input files and folder structure.

This notebook is organized into two main sections:
1. Loading the data files
2. Performing calculations

## Case study

- **Region:** Province of Quebec, Canada
- **Year:** 2021
- **Impact studied:** Climate change short-term
- **Available socioeconomic and demographic (SED) factors:** Income, Household type, Region and Tenure
                                                             

## Required libraries

In [1]:
import pandas as pd
import os 

## 1. Loading the data files

The following parameters define the impact category, SED factor, study year, and unit of measurement. Modify these values to adapt the notebook to a different impact category, SED factor, or study year.


In [2]:
REGION = "Quebec"
IMPACT = "CC"          
FACTOR = "Income"     
YEARS  = [2021] 
UNIT_BASE = "kg CO2 eq (short)"
households_per_factor = ["Q1", "Q2", "Q3", "Q4", "Q5"]

### Emissions per IO Sector

This file corresponds to the output generated in Step 1 (CC_Households_Quebec_2021.csv).

In [3]:
emissions = pd.read_csv(f"../Step_1/{IMPACT}_Households_{REGION}_{YEARS[0]}.csv", skiprows=1)
emissions = emissions.rename(columns={"Unnamed: 1": "IO_Sector", UNIT_BASE: UNIT_BASE})
emissions = emissions[["IO_Sector", UNIT_BASE]].dropna()
emissions

,IO_Sector,kg CO2 eq (short)
0,Private vehicle use,1.492214e+10
1,Heating and cooking,7.513357e+09
2,Motor gasoline,3.327073e+09
3,Prepared meals,2.653429e+09
4,Wearing apparel; furs (18),1.954002e+09
...,...,...
6089,Electricity by solar photovoltaic,9.954750e-05
6090,Electricity by solar photovoltaic,3.495714e-05
6091,Electricity by Geothermal,2.104386e-07
6092,Ethane,1.709717e-07


### Mapping between IO sectors and expenditure sectors

This file corresponds to the concordance table that extends the IO sector mapping from Step 1, linking each IO sector to its corresponding ISQ expenditure category and allocation proportion. 

In [4]:
mapping = pd.read_excel(f"{YEARS[0]}_{FACTOR}_corr.xlsx")
mapping = mapping[["IO_Sector", "Quebec_sector", "Proportion", "Consumption_category"]].dropna()
mapping

,IO_Sector,Quebec_sector,Proportion,Consumption_category
0,Artificial and synthetic fibres and filaments,Vêtements et accessoires,1.0,Clothing and accessories
1,Clothing accessories,Vêtements et accessoires,1.0,Clothing and accessories
2,Fabrics,Vêtements et accessoires,1.0,Clothing and accessories
3,"Fibre, yarn and thread",Vêtements et accessoires,1.0,Clothing and accessories
4,Footwear,Vêtements et accessoires,1.0,Clothing and accessories
...,...,...,...,...
405,Transportation services via pipelines,Transport public,1.0,Transportation
406,Urban transit services,Transport public,1.0,Transportation
407,Water freight transportation services,Transport public,1.0,Transportation
408,Water passenger transportation services,Transport public,1.0,Transportation


### Household-weighted expenditure shares by SED subfactor for each expenditure category

This file contains the household-weighted expenditure shares by SED subfactor for each ISQ expenditure category.


In [5]:
factor_df = pd.read_excel(f"spending_by_{FACTOR.lower()}.xlsx")
percent_cols = [f"{f}_percent" for f in households_per_factor]
factor_df = factor_df[["Sector"] + percent_cols]
factor_df = factor_df.rename(columns={"Sector": "Quebec_sector"})
factor_df

,Quebec_sector,Q1_percent,Q2_percent,Q3_percent,Q4_percent,Q5_percent
0,Ameublement et équipement ménagers,9.557745,14.058207,16.816092,23.553464,36.014492
1,Cotisations à des caisses de retraite ou de pe...,1.049938,4.792180,13.960370,27.880455,52.317057
2,Aliments achetés au magasin,11.764189,13.619567,17.742398,24.005697,32.868149
3,Aliments achetés au restaurant,9.187994,13.616258,16.337040,22.827277,38.031431
4,Dépenses courantes,12.420523,13.902262,17.567335,24.319701,31.790179
5,Dépenses diverses,8.099672,7.880421,21.688213,19.522296,42.809398
6,Logement,11.620756,13.296635,18.110457,24.767152,32.205000
7,Loisirs,7.191377,13.878045,15.751563,25.605228,37.573787
8,Primes d'assurance-emploi et d'assurance-paren...,1.718213,6.798249,16.429695,29.193108,45.860736
9,"Primes d'assurance-vie, d'assurances temporair...",11.055837,12.639067,15.334189,24.706902,36.264006


## 2. Performing calculations

The environmental impacts at the IO sector level are allocated to ISQ expenditure categories using the concordance table. The allocation is performed by multiplying the IO sector emissions by the corresponding proportion.


In [6]:
df_emissions_quebec = mapping.merge(emissions, on="IO_Sector", how="left")
df_emissions_quebec[f"Emissions_QC_{UNIT_BASE}"] = (df_emissions_quebec[UNIT_BASE] * df_emissions_quebec["Proportion"])
df_emissions_quebec

,IO_Sector,Quebec_sector,Proportion,Consumption_category,kg CO2 eq (short),Emissions_QC_kg CO2 eq (short)
0,Artificial and synthetic fibres and filaments,Vêtements et accessoires,1.0,Clothing and accessories,5.255602e+05,5.255602e+05
1,Artificial and synthetic fibres and filaments,Vêtements et accessoires,1.0,Clothing and accessories,1.635379e+05,1.635379e+05
2,Clothing accessories,Vêtements et accessoires,1.0,Clothing and accessories,6.854602e+06,6.854602e+06
3,Fabrics,Vêtements et accessoires,1.0,Clothing and accessories,1.615151e+07,1.615151e+07
4,Fabrics,Vêtements et accessoires,1.0,Clothing and accessories,7.155734e+05,7.155734e+05
...,...,...,...,...,...,...
6015,"Water transportation support, maintenance and ...",Transport public,1.0,Transportation,9.640528e+03,9.640528e+03
6016,"Water transportation support, maintenance and ...",Transport public,1.0,Transportation,5.207847e+02,5.207847e+02
6017,"Water transportation support, maintenance and ...",Transport public,1.0,Transportation,1.186316e+02,1.186316e+02
6018,"Water transportation support, maintenance and ...",Transport public,1.0,Transportation,9.405302e+01,9.405302e+01


In [7]:
df_grouped = df_emissions_quebec.groupby(
    ["Quebec_sector", "Consumption_category"], 
    as_index=False)[f"Emissions_QC_{UNIT_BASE}"].sum()
df_grouped

,Quebec_sector,Consumption_category,Emissions_QC_kg CO2 eq (short)
0,Aliments achetés au magasin,Food expenditures,1.803348e+10
1,Aliments achetés au restaurant,Recreation,6.105018e+07
2,Ameublement et équipement ménagers,"Household operations, furnishings and, equipment",3.201369e+09
3,Cotisations à des caisses de retraite ou de pe...,Miscellaneous expenditures,8.300356e+07
4,Dépenses courantes,"Household operations, furnishings and, equipment",5.632506e+09
5,Dépenses diverses,Miscellaneous expenditures,1.045647e+09
6,Frais directs de soins de santé défrayés par l...,Health and personal care,1.084554e+09
7,Logement,Recreation,2.501232e+08
8,Logement,Shelter,1.150391e+10
9,Loisirs,Recreation,1.059592e+09


### Impact allocation by SED subfactor

The allocated emissions are merged with the household-weighted expenditure shares to distribute the impacts across SED subfactors.


In [8]:
df_merge = pd.merge(df_grouped, factor_df, on="Quebec_sector", how="left")
df_merge

,Quebec_sector,Consumption_category,Emissions_QC_kg CO2 eq (short),Q1_percent,Q2_percent,Q3_percent,Q4_percent,Q5_percent
0,Aliments achetés au magasin,Food expenditures,1.803348e+10,11.764189,13.619567,17.742398,24.005697,32.868149
1,Aliments achetés au restaurant,Recreation,6.105018e+07,9.187994,13.616258,16.337040,22.827277,38.031431
2,Ameublement et équipement ménagers,"Household operations, furnishings and, equipment",3.201369e+09,9.557745,14.058207,16.816092,23.553464,36.014492
3,Cotisations à des caisses de retraite ou de pe...,Miscellaneous expenditures,8.300356e+07,1.049938,4.792180,13.960370,27.880455,52.317057
4,Dépenses courantes,"Household operations, furnishings and, equipment",5.632506e+09,12.420523,13.902262,17.567335,24.319701,31.790179
5,Dépenses diverses,Miscellaneous expenditures,1.045647e+09,8.099672,7.880421,21.688213,19.522296,42.809398
6,Frais directs de soins de santé défrayés par l...,Health and personal care,1.084554e+09,12.539856,17.278372,16.279330,23.091089,30.811353
7,Logement,Recreation,2.501232e+08,11.620756,13.296635,18.110457,24.767152,32.205000
8,Logement,Shelter,1.150391e+10,11.620756,13.296635,18.110457,24.767152,32.205000
9,Loisirs,Recreation,1.059592e+09,7.191377,13.878045,15.751563,25.605228,37.573787


In [9]:
for f in households_per_factor:
    df_merge[f"{IMPACT}_{f}"] = (
        df_merge[f"{f}_percent"] / 100
        * df_merge[f"Emissions_QC_{UNIT_BASE}"]
    )

In [10]:
impact_factor_cols = [f"{IMPACT}_{f}" for f in households_per_factor]

df_category = df_merge.groupby("Consumption_category")[impact_factor_cols].sum().reset_index()

df_category[f"Total_{IMPACT}_Category"] = df_category[impact_factor_cols].sum(axis=1)

for col in impact_factor_cols:
    total_col = df_category[col].sum()
    df_category[f"{col}_percent"] = (df_category[col] / total_col * 100)

total_values = df_category[impact_factor_cols + [f"Total_{IMPACT}_Category"]].sum()
total_percents = pd.Series(
    [100.0] * len(impact_factor_cols),
    index=[f"{col}_percent" for col in impact_factor_cols]
)

df_total_row = pd.DataFrame(
    {**{"Consumption_category": "Total"}, **total_values, **total_percents},
    index=[0]
)

df_category = pd.concat([df_category, df_total_row], ignore_index=True)
df_category


,Consumption_category,CC_Q1,CC_Q2,CC_Q3,CC_Q4,CC_Q5,Total_CC_Category,CC_Q1_percent,CC_Q2_percent,CC_Q3_percent,CC_Q4_percent,CC_Q5_percent
0,Clothing and accessories,3.187710e+08,6.246646e+08,8.350871e+08,1.210393e+09,1.581025e+09,4.569940e+09,4.012460,5.883933,6.115078,6.249737,5.946147
1,Food expenditures,2.121492e+09,2.456082e+09,3.199572e+09,4.329062e+09,5.927270e+09,1.803348e+10,26.703819,23.134685,23.429447,22.352666,22.292140
2,Health and personal care,2.486534e+08,2.914143e+08,3.594868e+08,5.064869e+08,6.968162e+08,2.102858e+09,3.129870,2.744933,2.632408,2.615193,2.620688
3,"Household operations, furnishings and, equipment",1.005565e+09,1.233101e+09,1.527826e+09,2.123842e+09,2.943541e+09,8.833875e+09,12.657333,11.615004,11.187788,10.966239,11.070495
4,Miscellaneous expenditures,1.011215e+08,1.041842e+08,2.600077e+08,2.621518e+08,5.422631e+08,1.269728e+09,1.272845,0.981347,1.903954,1.353594,2.039422
5,Recreation,1.108747e+08,1.886214e+08,2.221745e+08,3.471954e+08,5.018992e+08,1.370765e+09,1.395611,1.776690,1.626914,1.792708,1.887616
6,Shelter,1.336841e+09,1.529632e+09,2.083410e+09,2.849190e+09,3.704833e+09,1.150391e+10,16.827190,14.408137,15.256150,14.711498,13.933674
7,"Tobacco, alcohol and non-medical cannabis",2.871167e+08,3.427064e+08,4.728546e+08,6.243621e+08,5.833847e+08,2.310425e+09,3.614019,3.228071,3.462564,3.223830,2.194078
8,Transportation,2.414092e+09,3.846042e+09,4.695779e+09,7.114413e+09,1.010803e+10,2.817835e+10,30.386853,36.227200,34.385698,36.734536,38.015741
9,Total,7.944528e+09,1.061645e+10,1.365620e+10,1.936710e+10,2.658906e+10,7.817333e+10,100.000000,100.000000,100.000000,100.000000,100.000000


The following cells generate the output files for Steps 3 and 4. One file per consumption category is generated for Step 3, and one file per SED subfactor is generated for Step 4.


In [11]:
Exit_Folder = f"{IMPACT}_by_category"
os.makedirs(Exit_Folder, exist_ok=True)

for _, row in df_category.iterrows():
    category_name = row["Consumption_category"]
    impact_values = [row[f"{IMPACT}_{f}"] for f in households_per_factor]
    
    New_df = pd.DataFrame({
        FACTOR: list(households_per_factor),
        UNIT_BASE: impact_values,
    })
    
    short_name = category_name[:4]
    file_path = os.path.join(Exit_Folder, f"{short_name}.xlsx")
    New_df.to_excel(file_path, index=False)

In [12]:
type_folder = f"{IMPACT}_by_{FACTOR}"
os.makedirs(type_folder, exist_ok=True)

for f in households_per_factor:
    df_f = pd.DataFrame()
    df_f["Consumption_category"] = df_category["Consumption_category"]
    df_f[UNIT_BASE] = df_category[f"{IMPACT}_{f}"]
    
    output_path = os.path.join(type_folder, f"{IMPACT}_by_{f}.xlsx")
    df_f.to_excel(output_path, index=False)